In [ ]:
import pandas as pd

from _plot_utils import load_experiment_data, plot_radar, plot_roc, plot_lift

In [ ]:
df_meta = pd.read_csv("/opt/gpudata/ecg/echonext/EchoNext_metadata_100k.csv")
y_true = df_meta.loc[df_meta["split"] == "test", "shd_moderate_or_greater_flag"].to_numpy()

experiments = {
    "full": load_experiment_data("../runs/"),
    "32k": load_experiment_data("../runs-32k/"),
    "16k": load_experiment_data("../runs-16k/"),
    "8k": load_experiment_data("../runs-8k/"),
    "4k": load_experiment_data("../runs-4k/"),
    "2k": load_experiment_data("../runs-2k/"),
    "1k": load_experiment_data("../runs-1k/"),
    "512": load_experiment_data("../runs-512/"),
    "256": load_experiment_data("../runs-256/"),
}

print("Experiments found:")
for exp_subset, subset_results in experiments.items():
    print(f"\tSubset {exp_subset}:")
    for exp_name, exp_data in subset_results.items():
        print(f"\t\t{exp_name}: composite AUROC = {exp_data['composite_auroc']:.4f}")

In [ ]:
baselines_full = {a: (experiments[s][k], c, l, m) for a, s, k, c, l, m in [
    # ------------------------------------------------------------------------------------------------
    #        Alias            Subset        Key                         Color       Line    Marker
    # ------------------------------------------------------------------------------------------------
    ("columbia-minimodel",    "full", "echonext-minimodel",          "tab:purple",   "-",    None),
    ("resnet50-from-scratch", "full", "resnet50-2D",                 "tab:blue",     "-",    None),
    ("proto-from-scratch",    "full", "protoecgnet-echonext-fusion", "tab:orange",   "-",    None),
    ("proto-xfer-tuned",      "full", "protoecgnet-transfer-cat3",   "tab:green",    "-",    None),
    ("proto-xfer-logreg",     "full", "proto-all-logreg-pca64",      "tab:brown",    "-",    None),
    ("tabular-logreg",        "full", "logreg-unweighted",           "tab:red",      "-",    None),
    # ------------------------------------------------------------------------------------------------
]}

In [ ]:
plot_radar(baselines_full, title="Full-data Multilabel AUROCs")

In [ ]:
plot_roc(baselines_full, y_true, title="Full-data SHD ROC Curves")

In [ ]:
FULL_SIZE = 72475

aliases = {
    "echonext-minimodel": "columbia-minimodel",
    "resnet50-2D": "resnet50-from-scratch",
    "protoecgnet-echonext-fusion": "proto-from-scratch",
    "protoecgnet-transfer-cat3": "proto-xfer-tuned",
    "proto-all-logreg-pca64": "proto-xfer-logreg",
    "logreg-unweighted": "tabular-logreg",
}

palette = {
    "columbia-minimodel": "tab:purple",
    "resnet50-from-scratch": "tab:blue",
    "proto-from-scratch": "tab:orange",
    "proto-xfer-tuned": "tab:green",
    "proto-xfer-logreg": "tab:brown",
    "tabular-logreg": "tab:red",
}

df = pd.DataFrame.from_records(
    [
        {
            "Model": aliases[exp_name],
            "Train Size": FULL_SIZE if exp_subset == "full" else int(exp_subset.strip("k")) * 1024 if exp_subset.endswith("k") else int(exp_subset),
            "SHD (AUROC)": exp_data["composite_auroc"],
            "SHD (AUPRC)": exp_data["composite_auprc"],
            "Multilabel (AUROC)": exp_data["multilabel_avg_auroc"],
            "Multilabel (AUPRC)": exp_data["multilabel_avg_auprc"],
        }
        for exp_subset, subset_results in experiments.items()
        for exp_name, exp_data in subset_results.items()
        if exp_name in aliases
    ]
)

In [ ]:
plot_lift(
    data=df,
    metric="SHD (AUROC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="Binary SHD (AUROC)",
)

In [ ]:
plot_lift(
    data=df,
    metric="Multilabel (AUROC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="Multilabel Averaged SHD (AUROC)",
)

In [ ]:
plot_lift(
    data=df,
    metric="SHD (AUPRC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="Binary SHD (AUPRC)",
)

In [ ]:
plot_lift(
    data=df,
    metric="Multilabel (AUPRC)",
    baseline_model="columbia-minimodel",
    palette=palette,
    title="Multilabel Averaged SHD (AUPRC)",
)